# Empleos de analista de datos en el Perú

**Fuente:** [bumeran.com.pe](https://www.bumeran.com.pe) · **Herramienta:** Selenium

> **¿Dónde están hoy los empleos de analista de datos en el Perú, en qué
> modalidad, y qué herramientas piden?**

Bumeran no ofrece una descarga de sus avisos: hay que armar la base. Este
notebook recorre las búsquedas y lee **cada tarjeta del listado**. No hace
falta entrar a cada aviso: la tarjeta ya trae el puesto, la empresa, la
ubicación, la modalidad y la **descripción completa**.

**Genera:** `empleos_analista_datos_<fecha>.csv`, con una fila por aviso.
Correrlo toma **un par de minutos**.

## 0. Preparación

In [1]:
import math
import re
import time
from datetime import datetime

import pandas as pd
from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

El mismo trabajo se publica con varios nombres, así que se buscan las
variantes más comunes. Si agregas otras, **escríbelas sin tilde**: al probar,
Bumeran devolvió 0 resultados para *analista de información* y 19 para
*analista de informacion*.

In [2]:
TERMINOS_BUSQUEDA = ["analista de datos", "data analyst", "analista bi", "business intelligence"]

PAUSA_SEGUNDOS = 2       # entre página y página, para no saturar el servidor
HEADLESS = True          # False = ver la ventana de Chrome mientras trabaja

FECHA_EXTRACCION = datetime.now()
ARCHIVO_CSV = f"empleos_analista_datos_{FECHA_EXTRACCION:%Y-%m-%d}.csv"
ARCHIVO_CSV

'empleos_analista_datos_2026-10-23.csv'

## 1. Abrir el navegador

Bumeran dibuja los avisos con JavaScript, así que hace falta un navegador de
verdad. Una trampa: Chrome sin ventana se anuncia como **`HeadlessChrome`** y
Bumeran (que está detrás de Cloudflare) responde *"Sorry, you have been
blocked"*. Por eso se le pide que se presente como un Chrome normal.

In [3]:
opciones = Options()
if HEADLESS:
    opciones.add_argument("--headless=new")
opciones.add_argument("--window-size=1920,1080")
opciones.add_argument(
    "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0 Safari/537.36"
)

navegador = webdriver.Chrome(options=opciones)
print("Chrome", navegador.capabilities["browserVersion"], "abierto")

Chrome 156.0.8078.47 abierto


Antes de extraer se lee el `robots.txt`, donde el sitio dice qué rutas no
quiere que recorran los programas. Las búsquedas (`/empleos-busqueda-...`) no
están prohibidas.

In [4]:
navegador.get("https://www.bumeran.com.pe/robots.txt")
print(navegador.find_element(By.TAG_NAME, "body").text)

User-agent: *
Disallow: /*recientes=true
Disallow: /*relevantes=true
Disallow: /empleos.html/111
Disallow: /empleos.html/100
Disallow: /empleos-area-*/111
Disallow: /empleos-area-*/100
Disallow: /en-*/empleos.html/111
Disallow: /en-*/empleos.html/100
Disallow: /empleos-busquedaext-
Disallow: /empleos/aptitus/*
Disallow: /*?localidades=*
Disallow: /*&localidades=*

Sitemap: https://www.bumeran.com.pe/sitemap_avisos_bum.xml
Sitemap: https://www.bumeran.com.pe/sitemap_core_bum.xml
Sitemap: https://www.bumeran.com.pe/sitemap_empresas_bum.xml
Sitemap: https://www.bumeran.com.pe/sitemap_listados_ubicacion_bum.xml
Sitemap: https://www.bumeran.com.pe/sitemap_tags_bum.xml


## 2. Abrir una búsqueda

Bumeran escribe la búsqueda en la dirección:
`.../empleos-busqueda-analista-de-datos.html?page=2`. Así que no hace falta
escribir en el buscador ni hacer clic.

`WebDriverWait` espera hasta que aparezca el título «80 empleos para la
búsqueda…», que es la señal de que JavaScript terminó de dibujar los avisos.
Al probar apareció una trampa: a veces Bumeran responde con la página vacía
(el título dice solo «0»). Esperar más no sirve; hay que volver a pedirla.

In [5]:
RESULTADOS_LISTOS = EC.text_to_be_present_in_element((By.TAG_NAME, "h1"), "empleos para")


def abrir_busqueda(navegador, termino, pagina):
    termino_en_url = termino.replace(" ", "-")
    url = f"https://www.bumeran.com.pe/empleos-busqueda-{termino_en_url}.html?page={pagina}"
    for intento in [1, 2, 3]:
        navegador.get(url)
        try:
            WebDriverWait(navegador, 20).until(RESULTADOS_LISTOS)
            time.sleep(PAUSA_SEGUNDOS)
            return True
        except TimeoutException:
            if "you have been blocked" in navegador.page_source:
                return False        # insistir solo alarga el bloqueo
            print(f"    No cargó (intento {intento} de 3), se reintenta en 30 s")
            time.sleep(30)
    return False

In [6]:
abrir_busqueda(navegador, "analista de datos", 1)
navegador.find_element(By.TAG_NAME, "h1").text

'80 empleos para la búsqueda "analista de datos"'

## 3. Leer las tarjetas

Cada tarjeta es un enlace a `/empleos/<puesto>-<número>.html`. Se ubican **por
esa forma de la dirección** y no por su clase CSS (`sc-ekQYnd bAylLX`), que la
genera un programa y cambia cada vez que Bumeran actualiza su web.

In [7]:
def tarjetas_de_la_pagina(navegador):
    tarjetas = []
    for enlace in navegador.find_elements(By.CSS_SELECTOR, "a[href*='/empleos/']"):
        if re.search(r"-\d+\.html$", enlace.get_attribute("href")):
            tarjetas.append(enlace)
    return tarjetas


tarjetas = tarjetas_de_la_pagina(navegador)
print(len(tarjetas), "tarjetas en la página\n")
print(tarjetas[0].text[:400])

20 tarjetas en la página

Publicado ayer
Analista de Datos (Manteniento)
San Fernando - La buena Familia
3.8
En San Fernando nos encontramos en la búsqueda de un(a) Analista de Mantenimiento, responsable de analizar y optimizar la gestión del mantenimiento mediante el seguimiento de indicadores, el control de costos y recursos, el análisis de fallas y la implementación de mejoras digitales, contribuyendo a la eficiencia y 


Dentro de la tarjeta, cada dato va en un `<h3>`, pero **no siempre en la misma
posición**: si la empresa tiene calificación, aparece un `<h3>` extra con el
`3.8` y corre a los demás. Lo que no cambia es el **ícono** que va delante de
cada dato, y el ícono tiene nombre:

| Ícono | Dato |
|---|---|
| `icon-light-location-pin` | ubicación |
| `icon-light-office` | modalidad |
| `icon-bold-star` | calificación de la empresa |
| `icon-light-discapacity` | apto para personas con discapacidad |

Con XPath se pide *el primer `<h3>` que viene después del ícono*.

In [8]:
def dato_tras_icono(tarjeta, icono):
    # Si la tarjeta no trae ese ícono (empresa sin calificación, por ejemplo) devuelve None
    encontrados = tarjeta.find_elements(By.XPATH, f".//i[@name='{icono}']/following::h3[1]")
    if encontrados:
        return encontrados[0].text
    return None


for icono in ["icon-light-location-pin", "icon-light-office", "icon-bold-star", "icon-light-discapacity"]:
    print(f"{icono:<26} →  {dato_tras_icono(tarjetas[0], icono)}")

icon-light-location-pin    →  Surquillo, Lima
icon-light-office          →  Presencial
icon-bold-star             →  3.8
icon-light-discapacity     →  None


Todo junto en una función. El puesto es el único `<h2>`, la antigüedad y la
empresa son los dos primeros `<h3>` (esos sí son fijos), y la descripción es
el último párrafo: el primero, a veces, es la marca «Alta revisión de
perfiles».

In [9]:
def leer_tarjeta(tarjeta):
    url = tarjeta.get_attribute("href")
    titulos = tarjeta.find_elements(By.TAG_NAME, "h3")
    parrafos = tarjeta.find_elements(By.TAG_NAME, "p")
    return {
        "id_aviso": re.search(r"-(\d+)\.html$", url).group(1),
        "puesto": tarjeta.find_element(By.TAG_NAME, "h2").text,
        "empresa": titulos[1].text,
        "antiguedad": titulos[0].text,
        "calificacion_empresa": dato_tras_icono(tarjeta, "icon-bold-star"),
        "ubicacion": dato_tras_icono(tarjeta, "icon-light-location-pin"),
        "modalidad": dato_tras_icono(tarjeta, "icon-light-office"),
        "apto_discapacidad": dato_tras_icono(tarjeta, "icon-light-discapacity") is not None,
        "descripcion": parrafos[-1].text if parrafos else None,
        "url": url,
    }


pd.Series(leer_tarjeta(tarjetas[0]))

id_aviso                                                       1118492439
puesto                                    Analista de Datos (Manteniento)
empresa                                   San Fernando - La buena Familia
antiguedad                                                 Publicado ayer
calificacion_empresa                                                  3.8
ubicacion                                                 Surquillo, Lima
modalidad                                                      Presencial
apto_discapacidad                                                   False
descripcion             En San Fernando nos encontramos en la búsqueda...
url                     https://www.bumeran.com.pe/empleos/analista-de...
dtype: object

## 4. Recorrer todas las búsquedas

Bumeran muestra 20 avisos por página y el total está en el título. Con eso se
sabe cuántas páginas tiene cada búsqueda.

In [10]:
filas = []
inicio = time.time()

for termino in TERMINOS_BUSQUEDA:
    if not abrir_busqueda(navegador, termino, 1):
        print(f"  {termino}: no cargó, se salta")
        continue
    titulo = navegador.find_element(By.TAG_NAME, "h1").text
    total = int(titulo.split()[0].replace(".", ""))      # "80 empleos para..." -> 80
    paginas = math.ceil(total / 20)

    for pagina in range(1, paginas + 1):
        # La página 1 ya está abierta
        if pagina > 1:
            if not abrir_busqueda(navegador, termino, pagina):
                continue
        for tarjeta in tarjetas_de_la_pagina(navegador):
            fila = leer_tarjeta(tarjeta)
            fila["termino"] = termino
            filas.append(fila)

    print(f"  {termino:<24} {total:>4} avisos · {paginas} páginas")

print(f"\n{len(filas)} tarjetas leídas en {(time.time() - inicio) / 60:.1f} minutos")

  analista de datos          80 avisos · 4 páginas


  data analyst               13 avisos · 1 páginas


  analista bi                25 avisos · 2 páginas


  business intelligence      24 avisos · 2 páginas

142 tarjetas leídas en 1.0 minutos


In [11]:
navegador.quit()      # de aquí en adelante todo es pandas

## 5. Una fila por aviso

Un mismo aviso sale en varias búsquedas: un "Analista BI" aparece al buscar
*analista bi* y *business intelligence*. Se deja una sola fila y se anota en
qué búsquedas apareció.

In [12]:
listado = pd.DataFrame(filas)

# Si el orden cambia mientras se pasa de página, un aviso puede repetirse dentro de una misma búsqueda
listado = listado.drop_duplicates(subset=["id_aviso", "termino"])
terminos_por_aviso = listado.groupby("id_aviso")["termino"].agg("; ".join)

empleos = listado.drop_duplicates(subset="id_aviso").drop(columns="termino")
empleos["terminos_busqueda"] = empleos["id_aviso"].map(terminos_por_aviso)

print(len(listado), "tarjetas →", len(empleos), "avisos distintos")

142 tarjetas → 134 avisos distintos


## 6. Limpiar

- «Surquillo, Lima» son dos datos: distrito y departamento.
- La calificación llega como texto («3.8»): se pasa a número.
- La antigüedad («Publicado hace 4 días», «ayer», «hace 2 horas») se pasa a
  días. «Hace más de 15 días» queda vacío: no dice cuántos. Ojo: si dice
  *Actualizado*, cuenta desde la última actualización, no desde que se
  publicó.

In [13]:
partes = empleos["ubicacion"].str.split(", ")
empleos["distrito"] = partes.str[0]
empleos["departamento"] = partes.str[-1]

empleos["calificacion_empresa"] = pd.to_numeric(empleos["calificacion_empresa"])

empleos["dias_publicado"] = empleos["antiguedad"].str.extract(r"hace (\d+) días?")[0].astype("Int64")
empleos.loc[empleos["antiguedad"].str.contains("ayer"), "dias_publicado"] = 1
empleos.loc[empleos["antiguedad"].str.contains("hora|minuto"), "dias_publicado"] = 0

empleos["fecha_extraccion"] = FECHA_EXTRACCION.strftime("%Y-%m-%d %H:%M")

empleos[["ubicacion", "distrito", "departamento", "antiguedad", "dias_publicado"]].head()

,ubicacion,distrito,departamento,antiguedad,dias_publicado
0,"Surquillo, Lima",Surquillo,Lima,Publicado ayer,1
1,"Cercado De Lima, Lima",Cercado De Lima,Lima,Actualizado hace 2 días,2
2,"Lima, Lima",Lima,Lima,Actualizado hace 2 horas,0
3,"San Isidro, Lima",San Isidro,Lima,Actualizado hace 7 días,7
4,"San Isidro, Lima",San Isidro,Lima,Actualizado hace 3 días,3


### ¿Qué herramientas pide cada aviso?

Se buscan en el título y la descripción con expresiones regulares. Tres
cuidados:

- Buscar `excel` a secas cuenta «excelente». `\b` exige la palabra completa.
- **R** es una sola letra: solo cuenta en mayúscula y entre separadores
  («Python, R, SQL»). Es la única búsqueda que distingue mayúsculas.
- Se cuentan también «Phyton» y «Tableu», que aparecen mal escritos en
  algunos avisos.

In [14]:
HERRAMIENTAS = {
    "SQL": r"\bsql\b",
    "Python": r"python|phyton",
    "R": r"[\s,(/]R[\s,)/;.]|RStudio",
    "Power BI": r"power\s?bi",
    "Excel": r"\bexcel\b",
    "Tableau": r"tableau|tableu",
    "Qlik": r"qlik",
    "SAS": r"\bsas\b",
    "SPSS": r"spss",
    "Spark": r"spark",
    "AWS": r"\baws\b",
    "Azure": r"azure",
    "Google Cloud": r"\bgcp\b|google cloud|bigquery",
    "SAP": r"\bsap\b",
    "ETL": r"\betl\b",
}

texto_aviso = empleos["puesto"] + "\n" + empleos["descripcion"].fillna("")

columnas_herramientas = []
empleos["herramientas"] = ""          # una columna legible: "SQL, Python, Power BI"
for nombre, patron in HERRAMIENTAS.items():
    columna = "pide_" + nombre.lower().replace(" ", "_")
    columnas_herramientas.append(columna)
    # case=True (distinguir mayúsculas) solo para R
    empleos[columna] = texto_aviso.str.contains(patron, case=(nombre == "R"), regex=True)
    empleos.loc[empleos[columna], "herramientas"] += nombre + ", "

empleos["herramientas"] = empleos["herramientas"].str.rstrip(", ")
empleos["pide_ingles"] = texto_aviso.str.lower().str.contains(r"ingl[eé]s|english", regex=True)

empleos[["puesto", "herramientas"]].head(8)

,puesto,herramientas
0,Analista de Datos (Manteniento),"SAP, ETL"
1,CIBERTEC - Analista de Datos,"SQL, Python, Power BI"
2,Analista de Datos - Lima,"SQL, Excel, Tableau"
3,Analista de Datos Senior,"SQL, Python, R, AWS"
4,Analista de Datos,"SQL, Python, Power BI, Spark"
5,Analista de Datos,Power BI
6,Analista de Datos Sr. - Gloria,"SQL, Power BI, Excel, Tableau"
7,Analista de Datos y Analítica,"SQL, Python, Power BI, Excel"


## 7. Guardar

En `utf-8-sig`: esa firma al inicio hace que Excel muestre bien las tildes y
la ñ al abrir el archivo con doble clic. El nombre lleva la fecha porque los
avisos cambian todos los días.

In [15]:
COLUMNAS = (
    ["id_aviso", "puesto", "empresa", "calificacion_empresa", "modalidad",
     "distrito", "departamento", "ubicacion", "antiguedad", "dias_publicado",
     "apto_discapacidad", "herramientas", "pide_ingles"]
    + columnas_herramientas
    + ["terminos_busqueda", "descripcion", "url", "fecha_extraccion"]
)
empleos = empleos[COLUMNAS].reset_index(drop=True)

empleos.to_csv(ARCHIVO_CSV, index=False, encoding="utf-8-sig")
print(empleos.shape, "→", ARCHIVO_CSV)

(134, 32) → empleos_analista_datos_2026-10-23.csv


In [16]:
empleos.head()

,id_aviso,puesto,empresa,calificacion_empresa,modalidad,distrito,departamento,ubicacion,antiguedad,dias_publicado,...,pide_spark,pide_aws,pide_azure,pide_google_cloud,pide_sap,pide_etl,terminos_busqueda,descripcion,url,fecha_extraccion
0,1118492439,Analista de Datos (Manteniento),San Fernando - La buena Familia,3.8,Presencial,Surquillo,Lima,"Surquillo, Lima",Publicado ayer,1,...,False,False,False,False,True,True,analista de datos,En San Fernando nos encontramos en la búsqueda...,https://www.bumeran.com.pe/empleos/analista-de...,2026-10-23 19:00
1,1118487127,CIBERTEC - Analista de Datos,Laureate Perú,3.1,Híbrido,Cercado De Lima,Lima,"Cercado De Lima, Lima",Actualizado hace 2 días,2,...,False,False,False,False,False,False,analista de datos,¡Únete a Cibertec! En Cibertec contribuimos a ...,https://www.bumeran.com.pe/empleos/cibertec-an...,2026-10-23 19:00
2,1118482241,Analista de Datos - Lima,Empresa de Telecomunicaciones Olitel Ltda,NaN,Híbrido,Lima,Lima,"Lima, Lima",Actualizado hace 2 horas,0,...,False,False,False,False,False,False,analista de datos,Con más de 14 años de experiencia en el campo ...,https://www.bumeran.com.pe/empleos/analista-de...,2026-10-23 19:00
3,1118481791,Analista de Datos Senior,INDRA PERU,3.0,Presencial,San Isidro,Lima,"San Isidro, Lima",Actualizado hace 7 días,7,...,False,True,False,False,False,False,analista de datos,"En INDRA, estamos en búsqueda de un/a Analista...",https://www.bumeran.com.pe/empleos/analista-de...,2026-10-23 19:00
4,1118472492,Analista de Datos,Eureka Empresarial SAC,NaN,Presencial,San Isidro,Lima,"San Isidro, Lima",Actualizado hace 3 días,3,...,True,False,False,False,False,False,analista de datos,🚀 Nueva oportunidad en Eureka Consulting¿Te ap...,https://www.bumeran.com.pe/empleos/analista-de...,2026-10-23 19:00


## 8. Primeras respuestas

**¿Dónde están?**

In [17]:
empleos["departamento"].value_counts()

departamento
Lima                            121
Arequipa                          7
Gobierno Regional del Callao      2
Ica                               1
Huánuco                           1
Apurimac                          1
La Libertad                       1
Name: count, dtype: int64

**¿En qué modalidad?** (en porcentaje)

In [18]:
empleos["modalidad"].value_counts(normalize=True).mul(100).round(1)

modalidad
Presencial    62.7
Híbrido       35.1
Remoto         2.2
Name: proportion, dtype: float64

**¿Qué herramientas piden?** Porcentaje de avisos que menciona cada una.

In [19]:
empleos[columnas_herramientas + ["pide_ingles"]].mean().mul(100).round(1).sort_values(ascending=False)

pide_power_bi        69.4
pide_sql             58.2
pide_excel           54.5
pide_python          45.5
pide_etl             21.6
pide_ingles          18.7
pide_aws             12.7
pide_azure           11.9
pide_sap             11.9
pide_google_cloud    11.9
pide_tableau          9.7
pide_r                7.5
pide_spark            7.5
pide_qlik             1.5
pide_sas              0.0
pide_spss             0.0
dtype: float64

## Qué hicimos

| Paso | Para qué | Código clave |
|---|---|---|
| Presentarse como Chrome normal | Cloudflare bloquea a `HeadlessChrome` | `opciones.add_argument("user-agent=...")` |
| Leer el `robots.txt` | saber qué rutas no tocar | `navegador.get(".../robots.txt")` |
| Abrir la búsqueda por su dirección | no escribir en el buscador | `.../empleos-busqueda-{termino}.html?page={pagina}` |
| Esperar y reintentar | la página a veces llega vacía | `WebDriverWait(navegador, 20).until(RESULTADOS_LISTOS)` |
| Ubicar tarjetas por su dirección | las clases CSS cambian | `re.search(r"-\d+\.html$", href)` |
| Leer datos por su ícono | la posición de los `<h3>` cambia | `.//i[@name='icon-light-office']/following::h3[1]` |
| Una fila por aviso | el mismo aviso sale en varias búsquedas | `drop_duplicates(subset="id_aviso")` |
| Antigüedad en días | «hace 4 días» → 4 | `str.extract(r"hace (\d+) días?")` |
| Herramientas | palabra completa y errores de tipeo | `str.contains(r"\bexcel\b")` |
| Guardar | que Excel lea bien las tildes | `to_csv(..., encoding="utf-8-sig")` |